In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import yfinance as yf

In [2]:
# 1. Download Constituents Data (Auto-adjust turned off to preserve raw volume & prices)

tickers = {
    "HDFCBANK": "HDFCBANK.NS",
    "ICICIBANK": "ICICIBANK.NS",
    "RELIANCE": "RELIANCE.NS",
    "BHARTIARTL": "BHARTIARTL.NS",
    "LT": "LT.NS",
}

# Approximate free-float index weights (normalized to sum to 1.0)
# HDFC Bank ~10.54%, ICICI Bank ~8.81%,
# Reliance ~7.52%, Bharti Airtel ~5.12%, L&T ~4.17%
# Total ~36.16%

raw_weights = {
    "HDFCBANK": 0.1054,
    "ICICIBANK": 0.0881,
    "RELIANCE": 0.0752,
    "BHARTIARTL": 0.0512,
    "LT": 0.0417,
}

total_w = sum(raw_weights.values())
norm_weights = {stock: w / total_w for stock, w in raw_weights.items()}


# 2. Feature Engineering Function (Matches Baseline Exactly)
def calculate_technical_indicators(df):
  data = df.copy()
  if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

  data["Return"] = data["Close"].pct_change()
  data["SMA_10"] = data["Close"].rolling(window=10).mean()
  data["SMA_50"] = data["Close"].rolling(window=50).mean()

  # RSI
  delta = data["Close"].diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
  rs = gain / (loss + 1e-9)
  data["RSI"] = 100 - (100 / (1 + rs))

  # MACD
  ema_12 = data["Close"].ewm(span=12, adjust=False).mean()
  ema_26 = data["Close"].ewm(span=26, adjust=False).mean()
  data["MACD"] = ema_12 - ema_26

  data["Target_Return"] = data["Return"].shift(-1)
  return data.dropna()


# 3. Train Individual Models for Each Constituent
split_date = "2024-01-01"
features = ["Return", "SMA_10", "SMA_50", "RSI", "MACD", "Volume"]
predicted_returns_dict = {}

print("Training 5 constituent models...")
for symbol, ticker in tickers.items():
  print(f"-> Processing {symbol}...")
  raw_df = yf.download(
      ticker, start="2018-01-01", auto_adjust=False, progress=False
  )
  processed_df = calculate_technical_indicators(raw_df)

  train_mask = processed_df.index < split_date
  test_mask = processed_df.index >= split_date

  X_train = processed_df.loc[train_mask, features]
  y_train = processed_df.loc[train_mask, "Target_Return"]
  X_test = processed_df.loc[test_mask, features]

  scaler = MinMaxScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  # Train identical XGBoost configuration
  model = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=42)
  model.fit(X_train_scaled, y_train)

  # Store predictions mapped to test dates
  pred_ret = model.predict(X_test_scaled)
  predicted_returns_dict[symbol] = pd.Series(
      pred_ret, index=processed_df.loc[test_mask].index
  )

# Combine constituent predicted returns into a single DataFrame
constituent_preds_df = pd.DataFrame(predicted_returns_dict).dropna()

# 4. Synthesize Proxy Nifty Return via Weighted Aggregation
weighted_pred_return = np.zeros(len(constituent_preds_df))
for symbol in tickers.keys():
  weighted_pred_return += (
      constituent_preds_df[symbol].values * norm_weights[symbol]
  )

# 5. Evaluate Against Actual Nifty 50 Values
top_down_results = pd.read_csv(
    "top_down_results.csv", parse_dates=["Date"], index_col="Date"
)

# Align the index dates with the saved top-down test set
common_dates = top_down_results.index.intersection(constituent_preds_df.index[
    :-1
])
top_down_aligned = top_down_results.loc[common_dates]

# We need the actual Close price at day t to compute predicted Close at day t+1
nifty_actual_t = yf.download(
    "^NSEI", start="2023-12-01", auto_adjust=False, progress=False
)
if isinstance(nifty_actual_t.columns, pd.MultiIndex):
  nifty_actual_t.columns = nifty_actual_t.columns.get_level_values(0)

nifty_close_t = nifty_actual_t.loc[common_dates, "Close"].values
actual_close_next = top_down_aligned["Actual"].values
pred_price_A = top_down_aligned["Pred_TopDown"].values

# Calculate Model B predicted prices
subset_preds = constituent_preds_df.loc[common_dates]
aligned_weighted_return = np.zeros(len(common_dates))
for symbol in tickers.keys():
  aligned_weighted_return += (
      subset_preds[symbol].values * norm_weights[symbol]
  )

pred_price_B = nifty_close_t * (1 + aligned_weighted_return)

# Compute Model B Metrics
rmse_B = np.sqrt(mean_squared_error(actual_close_next, pred_price_B))
mape_B = mean_absolute_percentage_error(actual_close_next, pred_price_B) * 100

# Re-compute Model A on the exact matching dates for a strict 1-to-1 comparison
rmse_A = np.sqrt(mean_squared_error(actual_close_next, pred_price_A))
mape_A = mean_absolute_percentage_error(actual_close_next, pred_price_A) * 100

print("\n" + "=" * 45)
print("             HEAD-TO-HEAD RESULTS            ")
print("=" * 45)
print(f"Model A (Top-Down Baseline) : RMSE = {rmse_A:.2f} | MAPE = {mape_A:.2f}%")
print(f"Model B (Bottom-Up Top 5)   : RMSE = {rmse_B:.2f} | MAPE = {mape_B:.2f}%")
print("=" * 45)

# Save comparison results for plotting in your final paper
comparison_df = pd.DataFrame(
    {
        "Actual": actual_close_next,
        "TopDown_Pred": pred_price_A,
        "BottomUp_Pred": pred_price_B,
    },
    index=common_dates,
)
comparison_df.to_csv("comparison_results_v2(unnormal).csv")
print("Saved final results to 'comparison_results_v2(unormal).csv'")

Training 5 constituent models...
-> Processing HDFCBANK...


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

-> Processing ICICIBANK...


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

-> Processing RELIANCE...


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

... (2 outputs left out)

-> Processing LT...


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what


             HEAD-TO-HEAD RESULTS            
Model A (Top-Down Baseline) : RMSE = 228.20 | MAPE = 0.71%
Model B (Bottom-Up Top 5)   : RMSE = 251.39 | MAPE = 0.81%
Saved final results to 'comparison_results_v2(unormal).csv'


/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
/Users/aryanjain/Desktop/Subject/AIML/research/venv/lib/python3.13/site-packages/yfinance/multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

code save

In [11]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

# Sanity check so you get a clear message instead of a NameError
# assert "norm_models" in globals(), "Run the training loop cell first: norm_models is not defined"
# assert len(norm_models) == 5, f"Expected 5 constituents, found {len(norm_models)}"

bottom_norm_artifact = {
    "model_name": "Bottom-Up (Normalized)",
    "aggregation": "normalized",
    "constituents": norm_weights,
    "features": features,
}

joblib.dump(bottom_norm_artifact, "artifacts/bottom_norm_model.joblib")
print("Saved to artifacts/bottom_norm_model.joblib")
print("Constituents:", list(norm_weights.keys()))

Saved to artifacts/bottom_norm_model.joblib
Constituents: ['HDFCBANK', 'ICICIBANK', 'RELIANCE', 'BHARTIARTL', 'LT']
